In [1]:
import polars as pl
import datetime as dt
import os
from pathlib import Path

In [2]:
L2DATA_PATH = "D:/data/l2/"

In [3]:
def normalize_date(date: dt.date | dt.datetime | str) -> str:
    if isinstance(date, (dt.datetime, dt.date)):
        return date.strftime("%Y%m%d")
    return str(date).replace("-", "").replace(".", "").strip("/")

In [4]:
date = normalize_date('20260624')
filepath = Path(L2DATA_PATH)/"raw"/date

outpath = Path(L2DATA_PATH)/"proc"/date
outpath.mkdir(parents=True, exist_ok=True)

In [5]:
szcj_merged = pl.read_parquet(filepath/'szcj.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo']).rename({
    'LastPx':'Price',
    'LastQty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
])
szcj = szcj_merged.filter(pl.col('ExecType')==70).drop('ExecType')
szcancel = szcj_merged.filter(pl.col('ExecType')==52).drop('ExecType')

In [7]:
szcj.write_parquet(outpath/'szcj.pq',compression="gzip")
szcancel.write_parquet(outpath/'szcancel.pq',compression="gzip")

In [5]:
szwt = pl.read_parquet(filepath/'szwt.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo'])
szwt = szwt.with_columns([
    pl.col('Side').replace(49,1).replace(50,-1).cast(pl.Int8),
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f")
])
szwt

ChannelNo,ApplSeqNum,SecurityID,Price,OrderQty,Side,TransactTime,OrdType
i64,i64,i64,f64,i64,i8,time,i64
2012,1,2522,8.37,2500,1,09:15:00,50
2012,2,2167,19.25,13100,-1,09:15:00,50
2022,1,159201,1.222,592200,-1,09:15:00,50
2015,1,2217,2.28,5100,-1,09:15:00,50
2015,2,2217,2.28,5900,-1,09:15:00,50
…,…,…,…,…,…,…,…
2013,57966309,2602,13.18,3000,-1,14:59:59.990,50
2013,57966310,2007,12.79,1600,1,14:59:59.990,50
2013,57966311,300436,88.64,1000,1,14:59:59.990,50


In [6]:
szwt.write_parquet(outpath/'szwt.pq',compression="gzip")

In [5]:
sh = pl.read_parquet(filepath/'sh.pq').drop(['TradeMoney','TickBSFlag','LocalTime','SeqNo']).rename({
    'BizIndex':'ApplSeqNum',
    'Channel':'ChannelNo',
    'BuyOrderNO':'BidApplSeqNum',
    'SellOrderNO':'OfferApplSeqNum',
    'TickTime':'TransactTime',
    'Qty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f")
])
sh

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,BidApplSeqNum,OfferApplSeqNum,Price,OrderQty
i64,i64,i64,time,str,i64,i64,f64,i64
1,1,751028,06:00:00.270,"""S""",0,0,0.0,0
2,1,751900,06:00:00.270,"""S""",0,0,0.0,0
3,1,751004,06:00:00.270,"""S""",0,0,0.0,0
4,1,751019,06:00:00.270,"""S""",0,0,0.0,0
5,1,751992,06:00:00.270,"""S""",0,0,0.0,0
…,…,…,…,…,…,…,…,…
46340824,5,688757,15:05:01.010,"""S""",0,0,0.0,0
46340825,5,603257,15:05:01.010,"""S""",0,0,0.0,0
46340826,5,563900,15:05:01.010,"""S""",0,0,0.0,0


In [6]:
shcj = sh.filter(pl.col('Type')=='T').with_columns([
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side'),
]).drop('Type')
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,BidApplSeqNum,OfferApplSeqNum,Price,OrderQty,Side
i64,i64,i64,time,i64,i64,f64,i64,i32
877,1,603637,09:25:00.040,368731,45418,19.33,400,1
878,1,603637,09:25:00.040,368731,277909,19.33,400,1
879,1,603637,09:25:00.040,368731,358795,19.33,1400,1
880,1,603637,09:25:00.040,368731,360659,19.33,100,1
881,1,603637,09:25:00.040,368731,179544,19.33,200,1
…,…,…,…,…,…,…,…,…
45014691,3,603194,15:00:02.270,9584499,27168730,35.06,100,-1
45014692,3,603194,15:00:02.270,27144010,27168730,35.06,100,-1
45014693,3,603194,15:00:02.270,15679965,27168730,35.06,100,-1


In [7]:
shcj.write_parquet(outpath/'shcj.pq',compression="gzip")

In [8]:
shcancel = sh.filter(pl.col('Type')=='D').with_columns([
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side'),
]).drop('Type')

In [9]:
shcancel.write_parquet(outpath/'shcancel.pq',compression="gzip")

In [7]:
shwt_added = sh.filter(pl.col('Type')=='A').with_columns([
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side'),
]).drop('Type')

In [8]:
del sh

In [9]:
def RestoreOrder(wt, cj):
    
    cj = cj.sort(["ChannelNo", "ApplSeqNum", "SecurityID", "TransactTime"])

    # 剔除集合竞价
    cj_df = cj.filter(~((pl.col("TransactTime") < pl.time(9, 30, 0)) | (pl.col("TransactTime") >= pl.time(14, 57, 0))))

    # 1. 从成交表提取买卖双方订单并汇总
    cj_buy = cj_df.filter(pl.col('Side')==1).select([
        "ChannelNo", "BidApplSeqNum", "SecurityID", "OrderQty", "Price", "TransactTime", "Side"
    ]).rename({"BidApplSeqNum": "ApplSeqNum"})

    cj_sell = cj_df.filter(pl.col('Side')==-1).select([
        "ChannelNo", "OfferApplSeqNum", "SecurityID", "OrderQty", "Price", "TransactTime", "Side"
    ]).rename({"OfferApplSeqNum": "ApplSeqNum"})

    cj_all = pl.concat([cj_buy, cj_sell])
    cj_summary = cj_all.group_by(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]).agg([
        pl.sum("OrderQty"),
        pl.last("Price"),  # 一笔主动单可能同时吃掉多笔挂单
        pl.last("TransactTime")
    ])

    # 2. 使用反连接和半连接分离三种情况
    # 部分成交：同时存在于委托和成交（inner join）
    partial = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side", "OrderQty"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="inner"
    ).with_columns([
        (pl.col("OrderQty") + pl.col("OrderQty_right")).alias("OrderQty"),
        pl.lit("主动部分成交").alias("OrderStatus")
    ]).drop("OrderQty_right")

    # 未成交：存在于委托但不存在于成交（anti join）
    untouched = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns(
        pl.lit("挂单被动成交").alias("OrderStatus")
    )
    untouched = untouched.select(partial.columns)

    # 完全成交：存在于成交但不存在于委托（anti join）
    new = cj_summary.join(
        wt.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns([
        pl.lit("主动完全成交").alias("OrderStatus"),
    ])
    new = new.select(partial.columns)

    # 3. 合并所有订单
    init_order = pl.concat([partial, untouched, new])
    return init_order

In [10]:
shwt = RestoreOrder(shwt_added, shcj)
shwt

: 

In [ ]:
shwt.write_parquet(outpath/'shwt.pq',compression="gzip")